In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import json
import os
import torch
import torch.nn.functional as F

model_path = "../model_cache/intent_classifier_roberta/"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()

label_map_path = os.path.join(model_path, "label_map.json")
if os.path.exists(label_map_path):
    with open(label_map_path, "r", encoding="utf-8") as f:
        label_map = json.load(f)
else:
    label_map = {
        "similar_items": 0,
        "graph_pairing": 1,
        "color_variant": 2,
        "chit_chat": 3,
        "composite_intent": 4,
    }

print("model loaded")

d:\miniconda\envs\mRAG\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8227.34it/s]

model loaded


In [24]:
query = "I want to find a sleek, greenish khaki padded jacket,It features a zip and wind flap with press-studs"

inputs = tokenizer(query, return_tensors="pt", truncation=True, max_length=32)
inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}
with torch.no_grad():
    outputs = model(**inputs)

probs = F.softmax(outputs.logits, dim=-1)[0]
pred_idx = int(torch.argmax(probs).item())
confidence = float(probs[pred_idx].item())

id2label = {int(v): k for k, v in label_map.items()}
label = id2label.get(pred_idx, "unknown")

print(f"Q: '{query}'")
print(f"Pred: {label} | Confidence: {confidence:.4f}")

Q: 'I want to find a sleek, greenish khaki padded jacket,It features a zip and wind flap with press-studs'
Pred: composite_intent | Confidence: 0.9960
